# P2-2: Users’ perception analysis of cyberattack consequences using NLP

In [225]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import PorterStemmer
from itertools import groupby, permutations, product
from nltk.tokenize import RegexpTokenizer

import nltk
import spacy
from spacy import displacy
from spacy.symbols import ADJ, VERB, NOUN
from nltk.corpus import wordnet as wn
import re
import warnings
warnings.filterwarnings('ignore')

nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

print("All imports loaded.")

All imports loaded.


## Step 0: Load Data

In [226]:
df = pd.read_excel('data/all_data.xlsx')

# Drop the empty last column and metadata columns
doc_columns = df.columns[2:-1]  # columns C to AZ (50 documents)
documents = list(doc_columns)   # text descriptions are the column names
doc_vectors = df[doc_columns].values.T  # shape: (50 docs, 226 participants)

print(f"Number of documents: {len(documents)}")
print(f"Vector dimensionality (participants): {doc_vectors.shape[1]}")
print(f"\nFirst 3 documents:")
for d in documents[:3]:
    print(f"  - {d}")

Number of documents: 50
Vector dimensionality (participants): 226

First 3 documents:
  - The cyber-attacker accessed your computer files.
  - The cyber-attacker accessed your computer programs.
  - The cyber-attacker accessed your information stored in an Internet site.


## Step 1: Human-Based Similarity Matrix (Given Vectors)

In [227]:
# Compute pairwise Spearman correlation between all 50 document vectors
n_docs = len(documents)
human_sim_matrix = np.zeros((n_docs, n_docs))

for i in range(n_docs):
    for j in range(n_docs):
        if i == j:
            human_sim_matrix[i, j] = 1.0
        elif j > i:
            corr, _ = spearmanr(doc_vectors[i], doc_vectors[j])
            human_sim_matrix[i, j] = corr
            human_sim_matrix[j, i] = corr

human_sim_df = pd.DataFrame(human_sim_matrix, index=range(n_docs), columns=range(n_docs))
print("Human-Based Similarity Matrix (Spearman's rank correlation):")
print(f"Shape: {human_sim_df.shape}")
print(human_sim_df.iloc[:5, :5].round(3))

Human-Based Similarity Matrix (Spearman's rank correlation):
Shape: (50, 50)
       0      1      2      3      4
0  1.000  0.645  0.539  0.042  0.042
1  0.645  1.000  0.326  0.042 -0.029
2  0.539  0.326  1.000 -0.171 -0.171
3  0.042  0.042 -0.171  1.000  0.610
4  0.042 -0.029 -0.171  0.610  1.000


## Step 2: NLP Vectorization (BOW & TF-IDF)

In [228]:
"""
    Helper method to print the similarity matrices for NLP and WordNet methods.
"""
def print_sim_matrix_result(sim_matrix, method):
    print(f"Generated {len(sim_matrix)} {method} similarity matrices:\n")
    for key, mat in sim_matrix.items():
        print(f"  {key}: {mat.shape}")
        
    # Show a sample matrix
    sample_key = list(sim_matrix.keys())[0]
    print(f"\nSample matrix ({sample_key}), first 5x5:")
    print(pd.DataFrame(sim_matrix[sample_key]).iloc[:5, :5].round(3))

In [229]:
"""
    Helper method to setup the configurations for computing the similarity matrices for NLP and WordNet methods.
    Prints the number of configurations and each configuration combinations.
    Returns the configuration list.
"""
def config_similarity_method(method_type):
    configs = []
    if method_type == "NLP":
        for sw_included in [False, True]:       # No, Yes
            for stemming in [False, True]:       # No, Yes
                for ngram in [1, 2, 3]:          # Uni, Bi, Tri
                    ngram_label = {1: 'Uni', 2: 'Bi', 3: 'Tri'}[ngram]
                    configs.append({
                        'sw_included': sw_included,
                        'remove_stopwords': not sw_included,  # if SW included=No, we remove them
                        'stemming': stemming,
                        'ngram': ngram,
                        'label': f"SW_Inc={'Yes' if sw_included else 'No'}_Stem={'Yes' if stemming else 'No'}_{ngram_label}"
                    })
    elif method_type == "WordNet":
        for sw_included in [False, True]:       # No, Yes
            for lemmatization in [False, True]:       # No, Yes
                configs.append({
                    'sw_included': sw_included,
                    'remove_stopwords': not sw_included,  # if SW included=No, we remove them
                    'lemmatization': lemmatization,
                    'label': f"SW_Inc={'Yes' if sw_included else 'No'}_Lemma={'Yes' if lemmatization else 'No'}"
        })

    print(f"Total preprocessing configurations: {len(configs)}")
    for i, c in enumerate(configs):
        print(f"  {i+1}. {c['label']}")

    return configs

In [230]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))
"""
    Performs preprocessing of the text for NLP Vectorization.
    Lowercases the entire text and tokenizes it.
    If specified, remove stopwords and perform stemming on the tokenized text.
    Then concat the tokens and return it.
"""
def preprocess(text, remove_stopwords=False, apply_stemming=False):
    text = text.lower()
    tokens = re.findall(r'\b[a-z]+\b', text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    if apply_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

configs = config_similarity_method("NLP")

Total preprocessing configurations: 12
  1. SW_Inc=No_Stem=No_Uni
  2. SW_Inc=No_Stem=No_Bi
  3. SW_Inc=No_Stem=No_Tri
  4. SW_Inc=No_Stem=Yes_Uni
  5. SW_Inc=No_Stem=Yes_Bi
  6. SW_Inc=No_Stem=Yes_Tri
  7. SW_Inc=Yes_Stem=No_Uni
  8. SW_Inc=Yes_Stem=No_Bi
  9. SW_Inc=Yes_Stem=No_Tri
  10. SW_Inc=Yes_Stem=Yes_Uni
  11. SW_Inc=Yes_Stem=Yes_Bi
  12. SW_Inc=Yes_Stem=Yes_Tri


In [231]:
# Generate all 24 NLP similarity matrices (12 configs x 2 vectorizers)
nlp_similarity_matrices = {}

for config in configs:
    processed_docs = [preprocess(doc, config['remove_stopwords'], config['stemming']) for doc in documents]
    ngram_range = (config['ngram'], config['ngram'])
    
    for vec_name, VectorizerClass in [('BOW', CountVectorizer), ('TF-IDF', TfidfVectorizer)]:
        vectorizer = VectorizerClass(ngram_range=ngram_range)
        vectors = vectorizer.fit_transform(processed_docs)
        
        # Pairwise cosine similarity matrix
        sim_matrix = cosine_similarity(vectors)
        
        key = f"{vec_name}_{config['label']}"
        nlp_similarity_matrices[key] = sim_matrix


print_sim_matrix_result(nlp_similarity_matrices, "NLP")

Generated 24 NLP similarity matrices:

  BOW_SW_Inc=No_Stem=No_Uni: (50, 50)
  TF-IDF_SW_Inc=No_Stem=No_Uni: (50, 50)
  BOW_SW_Inc=No_Stem=No_Bi: (50, 50)
  TF-IDF_SW_Inc=No_Stem=No_Bi: (50, 50)
  BOW_SW_Inc=No_Stem=No_Tri: (50, 50)
  TF-IDF_SW_Inc=No_Stem=No_Tri: (50, 50)
  BOW_SW_Inc=No_Stem=Yes_Uni: (50, 50)
  TF-IDF_SW_Inc=No_Stem=Yes_Uni: (50, 50)
  BOW_SW_Inc=No_Stem=Yes_Bi: (50, 50)
  TF-IDF_SW_Inc=No_Stem=Yes_Bi: (50, 50)
  BOW_SW_Inc=No_Stem=Yes_Tri: (50, 50)
  TF-IDF_SW_Inc=No_Stem=Yes_Tri: (50, 50)
  BOW_SW_Inc=Yes_Stem=No_Uni: (50, 50)
  TF-IDF_SW_Inc=Yes_Stem=No_Uni: (50, 50)
  BOW_SW_Inc=Yes_Stem=No_Bi: (50, 50)
  TF-IDF_SW_Inc=Yes_Stem=No_Bi: (50, 50)
  BOW_SW_Inc=Yes_Stem=No_Tri: (50, 50)
  TF-IDF_SW_Inc=Yes_Stem=No_Tri: (50, 50)
  BOW_SW_Inc=Yes_Stem=Yes_Uni: (50, 50)
  TF-IDF_SW_Inc=Yes_Stem=Yes_Uni: (50, 50)
  BOW_SW_Inc=Yes_Stem=Yes_Bi: (50, 50)
  TF-IDF_SW_Inc=Yes_Stem=Yes_Bi: (50, 50)
  BOW_SW_Inc=Yes_Stem=Yes_Tri: (50, 50)
  TF-IDF_SW_Inc=Yes_Stem=Yes_Tri: (50, 5

# Step 3 - WordNet-Based Similarity Matrices

In [232]:
"""
    Maps Treebank tags to WordNet tags.
    If the Treebank tag don't match any WordNet tag listed here, discard it.
"""
def map_wordnet_pos(tb_tag):
    if tb_tag.startswith('J'):
        return wn.ADJ
    elif tb_tag.startswith('V'):
        return wn.VERB
    elif tb_tag.startswith('N'):
        return wn.NOUN
    elif tb_tag.startswith('R'):
        return wn.ADV
    else:
        return None

In [233]:
parts_speech = spacy.load("en_core_web_sm")
"""
    Performs preprocessing of the text using WordNet
    Performs POS tagging and converts these tags into WordNet tags. 
    Remove any stopwords if remove_stopwords is set to True
    Then look up the synset using the lemma or text depending if apply_lemmatization is set to True
    Returns the most frequent synset for each word in the text.
"""
def preprocess_wordnet(text, remove_stopwords=False, apply_lemmatization=False):
    text_pos = parts_speech(text)
    processed_synsets = []

    for token in text_pos:
        if remove_stopwords and token.is_stop:
            continue

        lookup_form = token.lemma_ if apply_lemmatization else token.text 
        wordnet_tag = map_wordnet_pos(token.tag_)

        if wordnet_tag:
            synsets = wn.synsets(lookup_form, pos=wordnet_tag)

            if synsets:
                processed_synsets.append(synsets[0])

    return processed_synsets

wn_configs = config_similarity_method("WordNet")

Total preprocessing configurations: 4
  1. SW_Inc=No_Lemma=No
  2. SW_Inc=No_Lemma=Yes
  3. SW_Inc=Yes_Lemma=No
  4. SW_Inc=Yes_Lemma=Yes


In [234]:
"""
    Computes the sentence similarities of the two synsets.
    For every lemma/word (depending on if we did lemmaization or not) in the smaller sentence, 
    Find the highest similarity score in respect to the larger sentence and keep tracking of the index of the lemmas/words in larger sentence.
    Once we get the highest similarity score, add it to the total similarity score and append the index where the highest similarity score was found. 
    This is to ensure we don't use the same lemma/word combination when finding the highest similarity score.
    Returns the average of the highest similarity score. 
"""
def sentence_similarity(synsets1, synsets2):
    similarity_score = 0
    similarity_count = 0

    used_idxes = set()

    for s1 in synsets1:
        max_similarity = 0
        best_idx = -1
        for idx, s2 in enumerate(synsets2):
            if idx in used_idxes:
                continue

            similarity = s1.path_similarity(s2)
            max_similarity = max(max_similarity, similarity)

            if similarity > max_similarity:
                best_idx = i

        if best_idx != -1:    
            similarity_score += max_similarity
            used_idxes.add(best_idx)
            similarity_count += 1

    return similarity_score / similarity_count if similarity_count > 0 else 0

In [235]:
"""
    Computes the symmetric sentence similarity between two synsets.
    Gets the highest similarity score for every word in synsets1 to the word in synsets2
    Do the same process with for every word in synsets2 to the word in synsets1 for symmetry as they may not equal to each other.
    Divide by 2 to get the mean of the highest similarity scores.
"""
def sym_sentence_similarity(synsets1, synsets2):
    sym_syn1 = sentence_similarity(synsets1, synsets2)
    sym_syn2 = sentence_similarity(synsets2, synsets1)

    return (sym_syn1 + sym_syn2) / 2

In [236]:
# Generate all 4 Wordnet similarity matrices (4 configs x 1 method (Greedy Lemma Aligning Overlap/GLAO))
wordnet_similarity_matrices = {}

for wn_config in wn_configs:
    processed_docs_wn = [preprocess_wordnet(doc, wn_config['remove_stopwords'], wn_config['lemmatization']) for doc in documents]
    n_processed_docs_wn = len(processed_docs_wn)
    wn_sim_matrix = np.zeros((n_processed_docs_wn, n_processed_docs_wn))

    # Pairwise symmetric similarity matrix
    for i in range(n_processed_docs_wn):
        for j in range(n_processed_docs_wn):
            if i == j:
                wn_sim_matrix[i, j] = 1.0
            elif j > i:
                wn_sim_matrix[i, j] = sym_sentence_similarity(processed_docs_wn[i], processed_docs_wn[j])   

    key = f"{wn_config['label']}"
    wordnet_similarity_matrices[key] = sim_matrix

print_sim_matrix_result(wordnet_similarity_matrices, "WordNet")

Generated 4 WordNet similarity matrices:

  SW_Inc=No_Lemma=No: (50, 50)
  SW_Inc=No_Lemma=Yes: (50, 50)
  SW_Inc=Yes_Lemma=No: (50, 50)
  SW_Inc=Yes_Lemma=Yes: (50, 50)

Sample matrix (SW_Inc=No_Lemma=No), first 5x5:
       0      1      2      3      4
0  1.000  0.750  0.355  0.013  0.013
1  0.750  1.000  0.349  0.013  0.203
2  0.355  0.349  1.000  0.010  0.010
3  0.013  0.013  0.010  1.000  0.093
4  0.013  0.203  0.010  0.093  1.000


# Step 4: compare NL Matrices with Human Matrix

## Extract half of the human matrix

In [237]:
# Extract the upper triangle of the Human Similarity Matrix
# k=1 excludes the diagonal
human_upper_tri = human_sim_matrix[np.triu_indices(n_docs, k=1)]

# Combine wordnet, BOW, and TF_IDF all in one
results = []

## Extract half of each matrix for BOW, TF-IDF, and WordNet. Then compare to the human matrix

In [238]:
# Extract half and find perason correlation score for BOW/TF-IDF
for label, matrix in nlp_similarity_matrices.items():
    nlp_upper_tri = matrix[np.triu_indices(n_docs, k=1)]
    
    corr, p_value = pearsonr(human_upper_tri, nlp_upper_tri)
    
    results.append({
        'Method': label,
        'Pearson_Correlation': corr,
    })

# Extract half and find pearson correlation score for WordNet
for label, matrix in wordnet_similarity_matrices.items():
    wn_upper_tri = matrix[np.triu_indices(n_docs, k=1)]
    
    corr, p_value = pearsonr(human_upper_tri, wn_upper_tri)
    
    results.append({
        'Method': label,
        'Pearson_Correlation': corr,
    })

## Sort results by pearson score and also finding the best for each technique

In [239]:
# Push to dataframe then sort by PC score
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(by='Pearson_Correlation', ascending=False).reset_index(drop=True)

print(comparison_df[['Method', 'Pearson_Correlation']])

# Find highest for each technique
best_bow_method = None
best_bow_score = -1.0
best_bow_found = False

best_idf_method = None
best_idf_score = -1.0
best_idf_found = False

best_wn_method = None
best_wn_score = -1.0
best_wn_found = False

for index, row in comparison_df.iterrows():

    #Just find the first occurence of each technique since the list is sorted by score
    if best_bow_found and best_idf_found and best_wn_found:
        break

    method = row['Method']
    score = row['Pearson_Correlation']
    
    if "BOW" in method and not best_bow_found:
        best_bow_method = method
        best_bow_score = score
        best_bow_found = True

    elif "TF-IDF" in method and not best_idf_found:
        best_idf_method = method
        best_idf_score = score
        best_idf_found = True

    #Wordnet doesnt have a special keyword, so we just make sure it is not the other two
    elif ("BOW" not in method) and ("TF-IDF" not in method) and not best_wn_found:
        best_wn_method = method
        best_wn_score = score
        best_wn_found = True
        
print()
print(f"Best BOW: {best_bow_score} ({best_bow_method})")
print(f"Best TF-IDF: {best_idf_score} ({best_idf_method})")
print(f"Best WordNet: {best_wn_score} ({best_wn_method})")

                            Method  Pearson_Correlation
0        BOW_SW_Inc=No_Stem=No_Uni             0.556295
1       BOW_SW_Inc=No_Stem=Yes_Uni             0.555727
2     TF-IDF_SW_Inc=No_Stem=No_Uni             0.551306
3    TF-IDF_SW_Inc=No_Stem=Yes_Uni             0.534726
4    TF-IDF_SW_Inc=Yes_Stem=No_Uni             0.495037
5      BOW_SW_Inc=Yes_Stem=Yes_Uni             0.494105
6       BOW_SW_Inc=Yes_Stem=No_Uni             0.493905
7   TF-IDF_SW_Inc=Yes_Stem=Yes_Uni             0.486581
8        BOW_SW_Inc=Yes_Stem=No_Bi             0.486143
9     TF-IDF_SW_Inc=Yes_Stem=No_Bi             0.485878
10      BOW_SW_Inc=Yes_Stem=Yes_Bi             0.483362
11   TF-IDF_SW_Inc=Yes_Stem=Yes_Bi             0.480147
12     TF-IDF_SW_Inc=No_Stem=No_Bi             0.445613
13    TF-IDF_SW_Inc=No_Stem=Yes_Bi             0.445410
14       BOW_SW_Inc=No_Stem=Yes_Bi             0.439998
15        BOW_SW_Inc=No_Stem=No_Bi             0.437720
16     BOW_SW_Inc=Yes_Stem=Yes_Tri             0